# Assignment 2: From Forecasts to Order Quantities — Multi-SKU Inventory Optimization

## Overview

In Module 2, you built time series forecasting models (ARMA, GBDT), learned quantile regression for a single SKU's newsvendor problem, and scaled to a multi-SKU portfolio with budget-constrained allocation. In this assignment, you will connect these pieces into a complete pipeline: build a forecasting model on real Walmart data, convert forecasts into SKU-level order quantities, evaluate portfolio-level inventory decisions, and analyze risk-return tradeoffs.

You will work with FOODS_3 item-level daily sales from Walmart store CA_1 (M5 dataset). The data, train/test split, cost structure, and helper functions are provided. You supply the modeling, decision logic, and analysis.

**What to submit:** This notebook, completed, with all code cells run and all written answers filled in. Verify with Kernel → Restart & Run All before submitting.

---
## Setup

Run the cell below to import libraries and set global parameters. **Do not modify this cell.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from scipy.optimize import minimize
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Setup complete.')

---
## Data Loading and Preparation

The cell below loads the M5 data, engineers features, and creates the train/test split. **Read it carefully but do not modify it.** You will build on top of this pipeline in the rest of the assignment.

**Forecasting setup:** All models in this assignment produce **1-day-ahead** predictions. Each test-day forecast uses realized values for lag and rolling features (i.e., we know yesterday's actual sales when predicting today). This corresponds to a daily replenishment setting where the store places orders each morning. If the procurement lead time were longer (e.g., 7 days), lag features within the lead time would not be available at decision time, and multi-step forecasting methods would be needed.

In [ ]:
# ── Load data ──────────────────────────────────────────────
df = pd.read_csv("sales_all_sku.csv", parse_dates=["date"])
df = df.sort_values(["item_id", "date"]).reset_index(drop=True)

n_skus = df["item_id"].nunique()
skus = df["item_id"].unique()
print(f"SKUs: {n_skus}  |  Date range: {df['date'].min().date()} to {df['date'].max().date()}")

# ── Per-SKU summary statistics ─────────────────────────────
sku_stats = df.groupby("item_id")["sales"].agg(["mean", "std", "median"]).rename(
    columns={"mean": "avg_sales", "std": "std_sales", "median": "med_sales"}
)
sku_stats["cv"] = sku_stats["std_sales"] / sku_stats["avg_sales"].replace(0, np.nan)
sku_stats["zero_frac"] = df.groupby("item_id")["sales"].apply(lambda x: (x == 0).mean())

# ── Day-of-week demand shares ──────────────────────────────
dow_shares = (
    df.groupby(["item_id", df["date"].dt.dayofweek])["sales"]
    .sum()
    .unstack(fill_value=0)
)
dow_shares = dow_shares.div(dow_shares.sum(axis=1), axis=0)
dow_shares.columns = [f"dow_share_{d}" for d in dow_shares.columns]

sku_features = sku_stats[["avg_sales", "cv", "zero_frac"]].copy()
sku_features = sku_features.join(dow_shares).fillna(0)

# ── Volume tiers (tercile cutoffs) ─────────────────────────
terciles = sku_features["avg_sales"].quantile([1/3, 2/3])
low_cut, high_cut = terciles.iloc[0], terciles.iloc[1]
sku_features["volume_tier"] = sku_features["avg_sales"].apply(
    lambda x: "low" if x <= low_cut else ("mid" if x <= high_cut else "high")
)

print(f"\nVolume tier cutoffs: low <= {low_cut:.3f} < mid <= {high_cut:.3f} < high")
print(sku_features["volume_tier"].value_counts().sort_index())

# ── Feature engineering ────────────────────────────────────
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["has_event"] = (df["event_name_1"] != "no_event").astype(int)

for lag in [1, 7, 14]:
    df[f"lag_{lag}"] = df.groupby("item_id")["sales"].shift(lag)

for window in [7, 14]:
    df[f"rolling_mean_{window}"] = df.groupby("item_id")["sales"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    df[f"rolling_std_{window}"] = df.groupby("item_id")["sales"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).std()
    )

# Merge static features
df = df.merge(sku_features[["volume_tier"]].reset_index(), on="item_id", how="left")
df["sku_code"] = df["item_id"].astype("category").cat.codes
for col in ["avg_sales", "cv", "zero_frac"]:
    if col not in df.columns:
        df = df.merge(sku_features[[col]].reset_index(), on="item_id", how="left")

# ── Train/test split: last 28 days ────────────────────────
TEST_DAYS = 28
cutoff_date = df["date"].max() - pd.Timedelta(days=TEST_DAYS - 1)

lag_cols = [c for c in df.columns if c.startswith("lag_")]
df_model = df.dropna(subset=lag_cols).copy()

train = df_model[df_model["date"] < cutoff_date].copy()
test = df_model[df_model["date"] >= cutoff_date].copy()

print(f"\nTrain: {len(train):,} rows  |  Test: {len(test):,} rows")
print(f"Cutoff: {cutoff_date.date()}")

# ── Feature sets ───────────────────────────────────────────
calendar_features = ["day_of_week", "day_of_month", "week_of_year", "is_weekend",
                     "month", "has_event", "snap_CA"]
lag_features = [c for c in df.columns if c.startswith("lag_")]
rolling_features = [c for c in df.columns if c.startswith("rolling_")]

local_features = calendar_features + lag_features + rolling_features
global_features = local_features + ["sku_code", "avg_sales", "cv", "zero_frac"]

TARGET = "sales"

print(f"Local features: {len(local_features)}  |  Global features: {len(global_features)}")

### Cost Structure

We assign heterogeneous cost parameters across SKUs. The newsvendor critical fractile for each SKU is:

$$\tau_i^* = \frac{p_i - c_i}{p_i - v_i}$$

where $p_i$ is selling price, $c_i$ is unit cost, and $v_i$ is salvage value. **Do not modify this cell.**

In [ ]:
# ── Heterogeneous cost structure ───────────────────────────
np.random.seed(RANDOM_STATE)

cost_df = sku_features[["avg_sales", "volume_tier"]].copy()
cost_df["price"] = np.where(
    cost_df["volume_tier"] == "high",
    np.random.uniform(2, 5, len(cost_df)),
    np.where(cost_df["volume_tier"] == "mid",
             np.random.uniform(4, 8, len(cost_df)),
             np.random.uniform(6, 12, len(cost_df)))
)
margin = np.random.uniform(0.2, 0.6, len(cost_df))
cost_df["cost"] = cost_df["price"] * (1 - margin)
salvage_frac = np.random.uniform(0, 0.5, len(cost_df))
cost_df["salvage"] = cost_df["cost"] * salvage_frac

cost_df["cu"] = cost_df["price"] - cost_df["cost"]       # underage cost
cost_df["co"] = cost_df["cost"] - cost_df["salvage"]      # overage cost
cost_df["tau_star"] = cost_df["cu"] / (cost_df["cu"] + cost_df["co"])

print("Cost structure summary:")
print(cost_df[["price", "cost", "salvage", "cu", "co", "tau_star"]].describe().round(3))

### Helper Functions

These functions are used throughout the assignment. **Do not modify this cell.**

In [ ]:
# ── Shared GBDT parameters ────────────────────────────────
GBDT_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "verbose": -1,
    "random_state": RANDOM_STATE,
    "n_estimators": 300,
}

def train_lgb(X_tr, y_tr, features, params=GBDT_PARAMS):
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr[features], y_tr)
    return model

def predict_lgb(model, X, features):
    return np.clip(model.predict(X[features]), 0, None)

def pinball_loss(y_true, q_pred, tau):
    residual = y_true - q_pred
    return np.mean(np.where(residual >= 0, tau * residual, (tau - 1) * residual))

def compute_profit(y_true, Q, price, cost, salvage):
    """Return total profit (scalar) over the period."""
    sold = np.minimum(y_true, Q)
    leftover = np.maximum(Q - y_true, 0)
    revenue = price * sold + salvage * leftover
    total_cost = cost * Q
    return (revenue - total_cost).sum()

def compute_daily_profit(y_true, Q, price, cost, salvage):
    """Return per-day profit array (not summed)."""
    sold = np.minimum(y_true, Q)
    leftover = np.maximum(Q - y_true, 0)
    return price * sold + salvage * leftover - cost * Q

# ── Select two focal SKUs for Parts 1-2 ───────────────────
sku_high = sku_features["avg_sales"].idxmax()
sku_sparse = sku_features["avg_sales"].idxmin()
print(f"High-volume SKU: {sku_high} (mean = {sku_features.loc[sku_high, 'avg_sales']:.2f})")
print(f"Sparse SKU:      {sku_sparse} (mean = {sku_features.loc[sku_sparse, 'avg_sales']:.3f}, "
      f"zero_frac = {sku_features.loc[sku_sparse, 'zero_frac']:.2f})")

---
## Part 1: Single-SKU Forecasting (15 points)

Work with the two focal SKUs defined above: one high-volume, one sparse. This part tests your understanding of the ARMA-to-GBDT progression from Lecture 1.

### 1a. AR Baseline (5 points)

For **each** of the two focal SKUs, fit an AR model using OLS with lags 1, 7, and 14 as predictors. Report RMSE on the test set for each SKU. Plot fitted vs. actual on the test period (one figure with two subplots, one per SKU).

💻 **Code required**

📊 **Report:** two-panel figure (fitted vs. actual for each SKU), RMSE for each SKU

✍️ **Written answer** (2-3 sentences): Looking at the residual plot for each SKU, is there evidence of nonlinearity or heteroskedasticity that the linear AR model misses?

In [ ]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

TARGET = "sales"

focal_skus = [sku_high, sku_sparse]
ar_results = []
pred_store = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=False)

for i, sku in enumerate(focal_skus):
    tr_sku = train[train["item_id"] == sku].copy()
    te_sku = test[test["item_id"] == sku].copy()

    ar_features = ["lag_1", "lag_7", "lag_14"]

    tr_sku = tr_sku.dropna(subset=ar_features + [TARGET]).copy()
    te_sku = te_sku.dropna(subset=ar_features + [TARGET]).copy()

    X_tr = sm.add_constant(tr_sku[ar_features], has_constant="add")
    y_tr = tr_sku[TARGET]

    X_te = sm.add_constant(te_sku[ar_features], has_constant="add")
    y_te = te_sku[TARGET]

    ar_model = sm.OLS(y_tr, X_tr).fit()
    y_pred = np.clip(ar_model.predict(X_te), 0, None)

    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    result_type = "high-volume" if sku == sku_high else "sparse"
    ar_results.append({
        "sku": sku,
        "type": result_type,
        "rmse": rmse,
        "n_train": len(tr_sku),
        "n_test": len(te_sku)
    })

    te_plot = te_sku.copy()
    te_plot["y_pred"] = y_pred
    te_plot["residual"] = te_plot[TARGET] - te_plot["y_pred"]
    pred_store[sku] = {"model": ar_model, "test_df": te_plot}

    xvals = np.arange(len(te_plot))
    axes[i].plot(xvals, te_plot[TARGET].values, label="Actual")
    axes[i].plot(xvals, te_plot["y_pred"].values, label="Fitted")
    axes[i].set_title(f"{result_type}: {sku}\nRMSE = {rmse:.4f}")
    axes[i].set_xlabel("Test period")
    axes[i].set_ylabel("Sales")
    axes[i].legend()

    print(f"\nSKU: {sku}")
    print(ar_model.params)
    print(f"Test RMSE: {rmse:.4f}")

plt.tight_layout()
plt.show()

ar_results_df = pd.DataFrame(ar_results)
display(ar_results_df)

*Your answer:* For the high-volume SKU, the fitted values follow the general pattern of demand, but they smooth out some large fluctuations, so the linear AR model may be missing more complex dynamics. For the sparse SKU, most demand is zero, and the model cannot capture the sudden nonzero spikes very well. This suggests that the sparse series is more nonlinear and harder to model with a simple linear AR baseline.




### 1b. GBDT Comparison (5 points)

For each focal SKU, fit a LightGBM regressor using `local_features` (lags + calendar + rolling). Report RMSE on the test set. Plot the residual ACF (up to 21 lags) for both the AR model and the GBDT, side by side (4 subplots: 2 SKUs x 2 models).

💻 **Code required**

📊 **Report:** RMSE for each SKU and model, 4-panel ACF figure

✍️ **Written answer** (2-3 sentences): Compare the ACF plots. Does either model leave more temporal structure in its residuals? If the patterns differ, what feature(s) or model properties explain the difference? If they are similar, what does that tell you about the value of the additional flexibility?

In [ ]:
# YOUR CODE HERE
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf

gbdt_results = []
gbdt_store = {}

for sku in [sku_high, sku_sparse]:
    tr = train[train["item_id"] == sku].copy()
    te = test[test["item_id"] == sku].copy()

    tr = tr.dropna(subset=local_features + [TARGET])
    te = te.dropna(subset=local_features + [TARGET])

    X_tr = tr[local_features]
    y_tr = tr[TARGET]
    X_te = te[local_features]
    y_te = te[TARGET]

    model = lgb.LGBMRegressor(
        objective="regression",
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_estimators=300,
        verbose=-1
    )

    model.fit(X_tr, y_tr)
    pred = np.clip(model.predict(X_te), 0, None)

    rmse = np.sqrt(mean_squared_error(y_te, pred))

    te2 = te.copy()
    te2["pred_gbdt"] = pred
    te2["resid_gbdt"] = te2[TARGET] - te2["pred_gbdt"]

    gbdt_store[sku] = {
        "model": model,
        "test_df": te2
    }

    gbdt_results.append({
        "sku": sku,
        "type": "high-volume" if sku == sku_high else "sparse",
        "rmse": rmse
    })

gbdt_results_df = pd.DataFrame(gbdt_results)
display(gbdt_results_df)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for r, sku in enumerate([sku_high, sku_sparse]):
    ar_resid = pred_store[sku]["test_df"]["residual"].dropna()
    gbdt_resid = gbdt_store[sku]["test_df"]["resid_gbdt"].dropna()

    plot_acf(ar_resid, lags=21, ax=axes[r, 0], title=f"{sku} - AR")
    plot_acf(gbdt_resid, lags=21, ax=axes[r, 1], title=f"{sku} - GBDT")

plt.tight_layout()
plt.show()

*Your answer:*GBDT does a little better than AR on both SKUs based on RMSE. Looking at the ACF plots, the residual autocorrelation is not dramatically different, but for the high-volume SKU the GBDT residuals seem slightly weaker, so the added features and nonlinear splits probably help. For the sparse SKU, both models still look similar because most observations are zero, which limits how much improvement the more flexible model can make.

### 1c. When Does Flexibility Help? (5 points)

Compare AR vs. GBDT RMSE on the high-volume and sparse SKU separately. Report the results in a table:

| Model | High-Volume RMSE | Sparse RMSE |
|-------|-----------------|-------------|
| AR    |                 |             |
| GBDT  |                 |             |
| Improvement (%) |       |             |

💻 **Code required** (table can be printed or formatted)

✍️ **Written answer** (2-3 sentences): Does GBDT improve over AR for both SKUs, or only one, or neither? If there is a difference, what explains it: nonlinearity, additional features (calendar, rolling stats), or something else? If GBDT does not help, why might a simpler linear model suffice here?

In [ ]:
# YOUR CODE HERE
comp = pd.DataFrame({
    "Model": ["AR", "GBDT"]
})

ar_high = ar_results_df.loc[ar_results_df["type"] == "high-volume", "rmse"].values[0]
ar_sparse = ar_results_df.loc[ar_results_df["type"] == "sparse", "rmse"].values[0]

gbdt_high = gbdt_results_df.loc[gbdt_results_df["type"] == "high-volume", "rmse"].values[0]
gbdt_sparse = gbdt_results_df.loc[gbdt_results_df["type"] == "sparse", "rmse"].values[0]

improve_high = (ar_high - gbdt_high) / ar_high * 100
improve_sparse = (ar_sparse - gbdt_sparse) / ar_sparse * 100

comp["High-Volume RMSE"] = [ar_high, gbdt_high]
comp["Sparse RMSE"] = [ar_sparse, gbdt_sparse]

improve_row = pd.DataFrame({
    "Model": ["Improvement (%)"],
    "High-Volume RMSE": [improve_high],
    "Sparse RMSE": [improve_sparse]
})

comp = pd.concat([comp, improve_row], ignore_index=True)
comp

*Your answer:*GBDT improves over AR for both SKUs, but the improvement is not very large. This probably comes from the extra local features, like calendar variables and rolling statistics, and also from the nonlinear flexibility of the tree model. Still, the gains are fairly modest, which suggests that the basic lag-based AR model already captures much of the main pattern in demand.

---
## Part 2: Single-SKU Quantile Regression (15 points)

Continue with the same two focal SKUs. This part tests your understanding of the quantile regression concepts from Lecture 2.

### 2a. Normal Approximation (5 points)

Using the GBDT mean model from Part 1b, compute the Normal approximation order quantity:

$$Q_i^* = \hat{\mu}_i + z_{\tau_i^*} \cdot \hat{\sigma}_i$$

where $\hat{\sigma}_i$ is the standard deviation of the GBDT's **training** residuals for that SKU, and $z_{\tau}$ is the standard Normal quantile.

For each focal SKU:
- Plot $Q^*$ and actual demand on the test period (same figure, two subplots)
- Report realized coverage (fraction of days where $Q^* \geq$ demand) and average daily profit

💻 **Code required**

📊 **Report:** two-panel figure, coverage and profit per SKU

In [ ]:
from scipy.stats import norm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

normal_results = []
normal_store = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, sku in enumerate([sku_high, sku_sparse]):
    tr = train[train["item_id"] == sku].copy()
    te = test[test["item_id"] == sku].copy()

    tr = tr.dropna(subset=local_features + [TARGET])
    te = te.dropna(subset=local_features + [TARGET])

    model = gbdt_store[sku]["model"]

    mu_tr = np.clip(model.predict(tr[local_features]), 0, None)
    mu_te = np.clip(model.predict(te[local_features]), 0, None)

    sigma_hat = (tr[TARGET] - mu_tr).std()

    tau = cost_df.loc[sku, "tau_star"]
    z = norm.ppf(tau)

    q_star = np.clip(mu_te + z * sigma_hat, 0, None)

    y_true = te[TARGET].values
    coverage = np.mean(q_star >= y_true)

    price = cost_df.loc[sku, "price"]
    cost = cost_df.loc[sku, "cost"]
    salvage = cost_df.loc[sku, "salvage"]

    daily_profit = compute_daily_profit(y_true, q_star, price, cost, salvage)
    avg_daily_profit = daily_profit.mean()

    normal_results.append({
        "sku": sku,
        "type": "high-volume" if sku == sku_high else "sparse",
        "tau_star": tau,
        "sigma_hat": sigma_hat,
        "coverage": coverage,
        "avg_daily_profit": avg_daily_profit
    })

    te2 = te.copy()
    te2["mu_hat"] = mu_te
    te2["q_star"] = q_star
    normal_store[sku] = te2

    axes[i].plot(te2[TARGET].values, label="Actual")
    axes[i].plot(te2["q_star"].values, label="Q*")
    axes[i].set_title(f"{sku}")
    axes[i].legend()

plt.tight_layout()
plt.show()

normal_results_df = pd.DataFrame(normal_results)
normal_results_df

### 2b. Quantile GBDT (5 points)

For each focal SKU, fit a LightGBM with `objective='quantile'` and `alpha=tau_star` (using that SKU's $\tau^*$ from `cost_df`). Plot $Q^*$ on the **same** figure as 2a (overlay on the Normal approximation). Report realized coverage and average daily profit.

💻 **Code required**

📊 **Report:** overlay figure (Normal + Quantile + actual for each SKU), coverage and profit per SKU

✍️ **Written answer** (2-3 sentences): Where in the test period do the two methods disagree most? What demand pattern drives the disagreement?

In [ ]:
# YOUR CODE HERE
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

quantile_results = []
quantile_store = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, sku in enumerate([sku_high, sku_sparse]):
    tr = train[train["item_id"] == sku].copy()
    te = test[test["item_id"] == sku].copy()

    tr = tr.dropna(subset=local_features + [TARGET])
    te = te.dropna(subset=local_features + [TARGET])

    tau = cost_df.loc[sku, "tau_star"]

    X_tr = tr[local_features]
    y_tr = tr[TARGET]
    X_te = te[local_features]
    y_te = te[TARGET].values

    model_q = lgb.LGBMRegressor(
        objective="quantile",
        alpha=tau,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_estimators=300,
        verbose=-1
    )

    model_q.fit(X_tr, y_tr)
    q_pred = np.clip(model_q.predict(X_te), 0, None)

    coverage = np.mean(q_pred >= y_te)

    price = cost_df.loc[sku, "price"]
    cost = cost_df.loc[sku, "cost"]
    salvage = cost_df.loc[sku, "salvage"]

    daily_profit = compute_daily_profit(y_te, q_pred, price, cost, salvage)
    avg_daily_profit = daily_profit.mean()

    te2 = te.copy()
    te2["q_quantile"] = q_pred
    quantile_store[sku] = te2

    quantile_results.append({
        "sku": sku,
        "type": "high-volume" if sku == sku_high else "sparse",
        "tau_star": tau,
        "coverage": coverage,
        "avg_daily_profit": avg_daily_profit
    })

    q_normal = normal_store[sku]["q_star"].values

    axes[i].plot(te[TARGET].values, label="Actual")
    axes[i].plot(q_normal, label="Normal Q*")
    axes[i].plot(q_pred, label="Quantile Q*")
    axes[i].set_title(f"{sku}")
    axes[i].set_xlabel("Test period")
    axes[i].legend()

plt.tight_layout()
plt.show()

quantile_results_df = pd.DataFrame(quantile_results)
quantile_results_df

*Your answer:*The two methods disagree much more on the sparse SKU than on the high-volume SKU. For FOODS_3_090, the Normal and quantile order quantities are fairly close for most of the test period, so the results are also similar. For FOODS_3_296, the Normal method keeps ordering a positive amount on many zero-demand days, while the quantile GBDT stays much closer to zero, so the disagreement is mainly driven by the zero-heavy intermittent demand pattern.

### 2c. Residual Diagnostics (5 points)

For each focal SKU:
1. Compute the GBDT mean model residuals on the **training** set
2. Plot a histogram of residuals with the fitted Normal density overlaid (use the residual mean and standard deviation)
3. Run a Shapiro-Wilk test (`scipy.stats.shapiro`) and report the test statistic and p-value

Present as a 2-panel figure (one per SKU).

💻 **Code required**

📊 **Report:** two-panel histogram with Normal overlay, Shapiro-Wilk results

✍️ **Written answer** (3-5 sentences): For which SKU does normality fail more severely? Connect this to the coverage results in 2a: does the SKU with worse normality also show worse coverage calibration under the Normal approximation? Explain the mechanism.

In [ ]:
# YOUR CODE HERE
from scipy.stats import norm, shapiro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

diag_results = []

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, sku in enumerate([sku_high, sku_sparse]):
    tr = train[train["item_id"] == sku].copy()
    tr = tr.dropna(subset=local_features + [TARGET])

    model = gbdt_store[sku]["model"]

    mu_tr = np.clip(model.predict(tr[local_features]), 0, None)
    resid = tr[TARGET].values - mu_tr

    m = resid.mean()
    s = resid.std()

    x = np.linspace(resid.min(), resid.max(), 300)
    y = norm.pdf(x, loc=m, scale=s)

    axes[i].hist(resid, bins=30, density=True, alpha=0.7)
    axes[i].plot(x, y, linewidth=2)
    axes[i].set_title(f"{sku}")
    axes[i].set_xlabel("Residual")
    axes[i].set_ylabel("Density")

    sw_stat, sw_p = shapiro(resid)

    diag_results.append({
        "sku": sku,
        "type": "high-volume" if sku == sku_high else "sparse",
        "resid_mean": m,
        "resid_std": s,
        "shapiro_stat": sw_stat,
        "p_value": sw_p
    })

plt.tight_layout()
plt.show()

diag_results_df = pd.DataFrame(diag_results)
diag_results_df

*Your answer:*Normality fails for both SKUs, but it is much worse for the sparse SKU FOODS_3_296. Its residuals are much more concentrated near zero and do not look like a smooth bell-shaped distribution, which is also reflected in the much lower Shapiro-Wilk statistic. This matches the result from 2a: the sparse SKU had worse coverage calibration under the Normal approximation, with realized coverage much higher than the target quantile. The reason is that when demand is zero-heavy and intermittent, the residual distribution is not well approximated by a Normal distribution, so the Normal correction can lead to over-ordering.

---
## Part 3: Multi-SKU Portfolio (30 points)

Scale from two SKUs to the full portfolio. This part tests your understanding of the multi-SKU modeling and quantile grid concepts from Lecture 3.

### 3a. Global Mean Model (5 points)

Train a single global GBDT using `global_features` (which include SKU identity and static attributes) to predict daily sales across all SKUs. Report:
- Aggregate RMSE on the test set
- RMSE broken out by volume tier (high / mid / low)

💻 **Code required**

📊 **Report:** RMSE table by tier

In [ ]:
# YOUR CODE HERE
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

train_g = train.dropna(subset=global_features + [TARGET]).copy()
test_g = test.dropna(subset=global_features + [TARGET]).copy()

global_model = lgb.LGBMRegressor(
    objective="regression",
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_estimators=300,
    verbose=-1
)

global_model.fit(train_g[global_features], train_g[TARGET])

test_g["pred_global"] = np.clip(global_model.predict(test_g[global_features]), 0, None)

overall_rmse = np.sqrt(mean_squared_error(test_g[TARGET], test_g["pred_global"]))

rmse_rows = []
rmse_rows.append({
    "group": "overall",
    "rmse": overall_rmse,
    "n": len(test_g)
})

for tier in ["high", "mid", "low"]:
    part = test_g[test_g["volume_tier"] == tier].copy()
    rmse = np.sqrt(mean_squared_error(part[TARGET], part["pred_global"]))
    rmse_rows.append({
        "group": tier,
        "rmse": rmse,
        "n": len(part)
    })

rmse_table = pd.DataFrame(rmse_rows)
rmse_table

### 3b. Improve the Forecast (10 points)

Implement **one** alternative to the global baseline. Choose either:
- **Volume-tier models**: one GBDT per tier (high / mid / low), using `local_features`
- **Cluster-based models**: run K-Means on the SKU feature matrix (`sku_features`), one GBDT per cluster. Choose K between 3 and 10; the exact value is not critical. Standardize features before clustering.

Compare your alternative against the global baseline:
1. Compute per-SKU RMSE for both approaches
2. Produce a scatter plot: global RMSE (x-axis) vs. your model RMSE (y-axis), colored by volume tier, with the 45-degree line marked
3. Report how many SKUs improve, how many worsen, and the average RMSE change by tier

💻 **Code required**

📊 **Report:** scatter plot, improvement summary table

✍️ **Written answer** (2-3 sentences): Does your alternative outperform the global baseline overall? If so, is the gain concentrated in sparse SKUs, high-volume SKUs, or both? If not, why might the global model be hard to beat, and what does this imply about the bias-variance tradeoff in this setting?

In [ ]:
# YOUR CODE HERE
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

train_alt = train.copy()
test_alt = test.copy()

if "volume_tier" not in train_alt.columns:
    train_alt = train_alt.merge(
        sku_features[["volume_tier"]],
        left_on="item_id",
        right_index=True,
        how="left"
    )

if "volume_tier" not in test_alt.columns:
    test_alt = test_alt.merge(
        sku_features[["volume_tier"]],
        left_on="item_id",
        right_index=True,
        how="left"
    )

train_alt = train_alt.dropna(subset=local_features + [TARGET, "volume_tier"]).copy()
test_alt = test_alt.dropna(subset=local_features + [TARGET, "volume_tier"]).copy()

tier_models = {}

for tier in ["high", "mid", "low"]:
    tr_t = train_alt[train_alt["volume_tier"] == tier].copy()

    model = lgb.LGBMRegressor(
        objective="regression",
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_estimators=300,
        verbose=-1
    )

    model.fit(tr_t[local_features], tr_t[TARGET])
    tier_models[tier] = model

test_alt["pred_tier"] = np.nan

for tier in ["high", "mid", "low"]:
    idx = test_alt["volume_tier"] == tier
    test_alt.loc[idx, "pred_tier"] = np.clip(
        tier_models[tier].predict(test_alt.loc[idx, local_features]),
        0,
        None
    )

if "pred_global" not in test_alt.columns:
    test_alt["pred_global"] = np.clip(
        global_model.predict(test_alt[global_features]),
        0,
        None
    )

sku_rows = []

for sku, part in test_alt.groupby("item_id"):
    y = part[TARGET].values
    rmse_global = np.sqrt(mean_squared_error(y, part["pred_global"]))
    rmse_tier = np.sqrt(mean_squared_error(y, part["pred_tier"]))
    tier = part["volume_tier"].iloc[0]

    sku_rows.append({
        "sku": sku,
        "volume_tier": tier,
        "rmse_global": rmse_global,
        "rmse_tier": rmse_tier,
        "rmse_change": rmse_tier - rmse_global,
        "improved": rmse_tier < rmse_global
    })

sku_compare = pd.DataFrame(sku_rows)

summary_rows = []
for tier in ["high", "mid", "low"]:
    part = sku_compare[sku_compare["volume_tier"] == tier]
    summary_rows.append({
        "tier": tier,
        "n_skus": len(part),
        "improved": int(part["improved"].sum()),
        "worsened": int((~part["improved"]).sum()),
        "avg_rmse_change": part["rmse_change"].mean()
    })

summary_table = pd.DataFrame(summary_rows)

overall_row = pd.DataFrame([{
    "tier": "overall",
    "n_skus": len(sku_compare),
    "improved": int(sku_compare["improved"].sum()),
    "worsened": int((~sku_compare["improved"]).sum()),
    "avg_rmse_change": sku_compare["rmse_change"].mean()
}])

summary_table = pd.concat([overall_row, summary_table], ignore_index=True)

plt.figure(figsize=(7, 7))

for tier in ["high", "mid", "low"]:
    part = sku_compare[sku_compare["volume_tier"] == tier]
    plt.scatter(part["rmse_global"], part["rmse_tier"], label=tier, alpha=0.7)

mx = max(sku_compare["rmse_global"].max(), sku_compare["rmse_tier"].max())
plt.plot([0, mx], [0, mx], linestyle="--")
plt.xlabel("Global RMSE")
plt.ylabel("Tier-model RMSE")
plt.legend()
plt.tight_layout()
plt.show()

display(summary_table)

*Your answer:*The volume-tier model does not outperform the global baseline overall. Although it improves more SKUs in the low-volume tier and has a slightly negative average RMSE change there, it performs worse on the high-volume tier and ends up slightly worse overall. This suggests that splitting the data by tier can help for sparse or low-volume SKUs, but the global model is hard to beat because pooling all SKUs gives more data and more stable estimates, which seems especially valuable for high-volume items.

### 3c. Quantile Grid (15 points)

Using `global_features`, train global GBDTs at $\tau = 0.1, 0.2, \ldots, 0.9$ with pinball loss (`objective='quantile'`, `alpha=tau`).

For each SKU:
1. Fix quantile crossings using isotonic regression (`sklearn.isotonic.IsotonicRegression`)
2. Interpolate to the SKU's $\tau_i^*$ from `cost_df`
3. Round to non-negative integers

*Note:* Training 9 global GBDTs on the full dataset may take 1-2 minutes. This is expected.

Report:
- Number of SKUs with crossings before the isotonic fix
- For one sparse SKU (use `sku_sparse`), produce a figure showing predicted quantiles at $\tau = 0.1, 0.3, 0.5, 0.7, 0.9$ overlaid on actual test sales

💻 **Code required**

📊 ***Note:* Training 9 global GBDTs on the full dataset may take 1-2 minutes. This is expected.

Report:** crossing count, quantile fan figure for sparse SKU

✍️ **Written answer** (2-3 sentences): For the sparse SKU, why are the lower quantile predictions flat at zero? Is this a model failure or an accurate representation of the conditional distribution?

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression

taus = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

train_q = train.dropna(subset=global_features + [TARGET]).copy()
test_q = test.dropna(subset=global_features + [TARGET]).copy()

# train 9 quantile models
for tau in taus:
    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=tau,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_estimators=100,
        verbose=-1
    )
    model.fit(train_q[global_features], train_q[TARGET])
    test_q[f"q_{tau:.1f}"] = np.clip(model.predict(test_q[global_features]), 0, None)

cross_before = 0
q_star_grid_all = []
sparse_plot_df = None

for sku in test_q["item_id"].unique():
    part = test_q[test_q["item_id"] == sku].copy()

    qmat = part[[f"q_{tau:.1f}" for tau in taus]].values
    fixed = np.zeros_like(qmat)

    # count crossing at SKU level
    has_cross = False
    for r in range(qmat.shape[0]):
        row = qmat[r, :]
        if np.any(np.diff(row) < 0):
            has_cross = True

        iso = IsotonicRegression(increasing=True, out_of_bounds="clip")
        fixed[r, :] = iso.fit_transform(taus, row)

    if has_cross:
        cross_before += 1

    tau_star = cost_df.loc[sku, "tau_star"]

    q_star_sku = []
    for r in range(fixed.shape[0]):
        val = np.interp(tau_star, taus, fixed[r, :])
        val = max(0, round(val))
        q_star_sku.append(val)

    q_star_grid_all.extend(q_star_sku)

    if sku == sku_sparse:
        sparse_plot_df = part.copy()
        for j, tau in enumerate(taus):
            sparse_plot_df[f"fixed_q_{tau:.1f}"] = fixed[:, j]

test_q["q_star_grid"] = q_star_grid_all

print("Number of SKUs with crossings before isotonic fix:", cross_before)

plt.figure(figsize=(12,5))
x = np.arange(len(sparse_plot_df))
plt.plot(x, sparse_plot_df[TARGET].values, label="Actual", linewidth=2)
for tau in [0.1, 0.3, 0.5, 0.7, 0.9]:
    plt.plot(x, sparse_plot_df[f"fixed_q_{tau:.1f}"].values, label=f"tau={tau}")
plt.title(f"Quantile fan for {sku_sparse}")
plt.xlabel("Test period")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

*Your answer:*For the sparse SKU, the lower quantiles are flat at zero because zero demand happens very frequently, so a large part of the conditional distribution is concentrated at zero. This is not really a model failure. It is actually a reasonable representation of intermittent demand: when the probability of zero demand is already above a low quantile level, the corresponding conditional quantile should stay at zero.

---
## Part 4: Portfolio Evaluation (20 points)

Evaluate the inventory decisions from Part 3 at the portfolio level.

### 4a. Coverage and Profit (5 points)

For **both** methods below, compute per-SKU realized coverage and portfolio profit:

**Method 1 (Normal approximation):** Using your mean model from Part 3 (alternative from 3b, or global from 3a), train a variance model: fit a global GBDT on the absolute residuals $|y - \hat{\mu}|$ from the training set. Then compute $Q_i^* = \hat{\mu}_i + z_{\tau_i^*} \cdot \hat{\sigma}_i$ for each SKU, clipped to non-negative integers.

**Method 2 (Quantile grid):** Use the orders from Part 3c.

Produce:
1. Per-SKU realized coverage (fraction of test days where $Q^* \geq$ demand)
2. Scatter plot: target $\tau_i^*$ (x-axis) vs. realized coverage (y-axis), colored by zero fraction, with the 45-degree line. Two panels, one per method.
3. Total portfolio profit for both methods, broken down by volume tier.

💻 **Code required**

📊 **Report:** coverage scatter plot (two panels), profit table by tier

In [ ]:
# YOUR CODE HERE
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

eval_df = test.copy()

if "volume_tier" not in eval_df.columns or "zero_frac" not in eval_df.columns:
    eval_df = eval_df.merge(
        sku_features[["volume_tier", "zero_frac"]],
        left_on="item_id",
        right_index=True,
        how="left"
    )

eval_df = eval_df.dropna(subset=global_features + [TARGET]).copy()

# method 1: normal approximation using global mean + variance model
eval_df["mu_hat"] = np.clip(global_model.predict(eval_df[global_features]), 0, None)

train_var = train.copy()
if "volume_tier" not in train_var.columns or "zero_frac" not in train_var.columns:
    train_var = train_var.merge(
        sku_features[["volume_tier", "zero_frac"]],
        left_on="item_id",
        right_index=True,
        how="left"
    )

train_var = train_var.dropna(subset=global_features + [TARGET]).copy()
train_var["mu_hat"] = np.clip(global_model.predict(train_var[global_features]), 0, None)
train_var["abs_resid"] = np.abs(train_var[TARGET] - train_var["mu_hat"])

var_model = lgb.LGBMRegressor(
    objective="regression",
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_estimators=100,
    verbose=-1
)

var_model.fit(train_var[global_features], train_var["abs_resid"])

eval_df["sigma_hat"] = np.clip(var_model.predict(eval_df[global_features]), 0, None)

z_map = cost_df["tau_star"].apply(lambda x: __import__("scipy").stats.norm.ppf(x)).to_dict()
eval_df["tau_star"] = eval_df["item_id"].map(cost_df["tau_star"])
eval_df["z_tau"] = eval_df["item_id"].map(z_map)

eval_df["q_normal"] = np.round(
    np.clip(eval_df["mu_hat"] + eval_df["z_tau"] * eval_df["sigma_hat"], 0, None)
).astype(int)

# method 2: quantile grid from part 3c
# this assumes test_q from 3c still exists and has q_star_grid
grid_orders = test_q[["item_id", "date", "q_star_grid"]].copy()
eval_df = eval_df.merge(grid_orders, on=["item_id", "date"], how="left")

# per-SKU coverage
rows = []

for sku, part in eval_df.groupby("item_id"):
    y = part[TARGET].values

    q1 = part["q_normal"].values
    q2 = part["q_star_grid"].values

    tau = cost_df.loc[sku, "tau_star"]
    price = cost_df.loc[sku, "price"]
    cost = cost_df.loc[sku, "cost"]
    salvage = cost_df.loc[sku, "salvage"]
    tier = part["volume_tier"].iloc[0]
    zero_frac = part["zero_frac"].iloc[0]

    cov1 = np.mean(q1 >= y)
    cov2 = np.mean(q2 >= y)

    prof1 = compute_daily_profit(y, q1, price, cost, salvage).sum()
    prof2 = compute_daily_profit(y, q2, price, cost, salvage).sum()

    rows.append({
        "sku": sku,
        "volume_tier": tier,
        "zero_frac": zero_frac,
        "tau_star": tau,
        "coverage_normal": cov1,
        "coverage_grid": cov2,
        "profit_normal": prof1,
        "profit_grid": prof2
    })

sku_eval = pd.DataFrame(rows)

# coverage scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(sku_eval["tau_star"], sku_eval["coverage_normal"], c=sku_eval["zero_frac"], alpha=0.7)
mx1 = max(sku_eval["tau_star"].max(), sku_eval["coverage_normal"].max())
axes[0].plot([0, 1], [0, 1], linestyle="--")
axes[0].set_title("Normal approximation")
axes[0].set_xlabel("Target tau*")
axes[0].set_ylabel("Realized coverage")

axes[1].scatter(sku_eval["tau_star"], sku_eval["coverage_grid"], c=sku_eval["zero_frac"], alpha=0.7)
axes[1].plot([0, 1], [0, 1], linestyle="--")
axes[1].set_title("Quantile grid")
axes[1].set_xlabel("Target tau*")
axes[1].set_ylabel("Realized coverage")

plt.tight_layout()
plt.show()

# profit by tier
profit_table = sku_eval.groupby("volume_tier")[["profit_normal", "profit_grid"]].sum().reset_index()

overall_row = pd.DataFrame([{
    "volume_tier": "overall",
    "profit_normal": sku_eval["profit_normal"].sum(),
    "profit_grid": sku_eval["profit_grid"].sum()
}])

profit_table = pd.concat([overall_row, profit_table], ignore_index=True)
profit_table

### 4b. Budget-Constrained Allocation (5 points)

Set the budget at **75%** of the unconstrained total procurement cost (using Method 2 orders).

Implement two allocation strategies:
1. **Proportional scaling**: multiply all orders by `budget / total_cost`
2. **Greedy marginal allocation**: rank SKUs by marginal expected profit per unit cost, allocate one unit at a time to the top-ranked SKU, repeat until budget is exhausted

Report portfolio profit under both strategies.

*Hint:* For efficiency, work with **total units per SKU** over the test period rather than allocating day by day. Compute each SKU's total unconstrained units and total cost, run the greedy loop at the SKU-total level, then distribute the allocated units back to daily orders proportionally to the unconstrained daily pattern.

💻 **Code required**

📊 **Report:** profit under proportional vs. greedy, improvement percentage

✍️ **Written answer** (2-3 sentences): Which SKUs does the greedy algorithm prioritize over proportional scaling, and why?

In [ ]:
# YOUR CODE HERE
import numpy as np
import pandas as pd

df = eval_df.copy()

sku_totals = df.groupby("item_id").agg(
    total_q=("q_star_grid", "sum")
).reset_index()

sku_totals["cost"] = sku_totals["item_id"].map(cost_df["cost"])
sku_totals["price"] = sku_totals["item_id"].map(cost_df["price"])
sku_totals["salvage"] = sku_totals["item_id"].map(cost_df["salvage"])
sku_totals["cu"] = sku_totals["price"] - sku_totals["cost"]
sku_totals["co"] = sku_totals["cost"] - sku_totals["salvage"]

total_cost_unconstrained = (sku_totals["total_q"] * sku_totals["cost"]).sum()
budget = 0.75 * total_cost_unconstrained
scale = budget / total_cost_unconstrained

# proportional
sku_totals["alloc_prop_total"] = np.floor(sku_totals["total_q"] * scale).astype(int)

# greedy
sku_totals["alloc_greedy_total"] = 0
sku_totals["remaining"] = sku_totals["total_q"].astype(int)

# simple marginal profit per unit cost proxy
sku_totals["score"] = sku_totals["cu"] / sku_totals["cost"]

budget_left = budget

while True:
    feasible = sku_totals[(sku_totals["remaining"] > 0) & (sku_totals["cost"] <= budget_left)]
    if len(feasible) == 0:
        break

    j = feasible["score"].idxmax()
    sku_totals.loc[j, "alloc_greedy_total"] += 1
    sku_totals.loc[j, "remaining"] -= 1
    budget_left -= sku_totals.loc[j, "cost"]

def spread_back(part, total_units, order_col):
    q = part[order_col].values.astype(float)
    out = np.zeros(len(part), dtype=int)

    if total_units <= 0 or q.sum() <= 0:
        return out

    w = q / q.sum()
    raw = total_units * w
    base = np.floor(raw).astype(int)
    rem = int(total_units - base.sum())

    if rem > 0:
        frac_idx = np.argsort(-(raw - base))[:rem]
        base[frac_idx] += 1

    return base

prop_rows = []
greedy_rows = []

for sku, part in df.groupby("item_id", sort=False):
    total_prop = int(sku_totals.loc[sku_totals["item_id"] == sku, "alloc_prop_total"].values[0])
    total_greedy = int(sku_totals.loc[sku_totals["item_id"] == sku, "alloc_greedy_total"].values[0])

    prop_alloc = spread_back(part, total_prop, "q_star_grid")
    greedy_alloc = spread_back(part, total_greedy, "q_star_grid")

    tmp1 = part.copy()
    tmp1["q_prop"] = prop_alloc
    prop_rows.append(tmp1)

    tmp2 = part.copy()
    tmp2["q_greedy"] = greedy_alloc
    greedy_rows.append(tmp2)

prop_df = pd.concat(prop_rows).sort_index()
greedy_df = pd.concat(greedy_rows).sort_index()

prop_profit = 0.0
greedy_profit = 0.0

for sku, part in prop_df.groupby("item_id"):
    y = part[TARGET].values
    q = part["q_prop"].values
    price = cost_df.loc[sku, "price"]
    cost = cost_df.loc[sku, "cost"]
    salvage = cost_df.loc[sku, "salvage"]
    prop_profit += compute_daily_profit(y, q, price, cost, salvage).sum()

for sku, part in greedy_df.groupby("item_id"):
    y = part[TARGET].values
    q = part["q_greedy"].values
    price = cost_df.loc[sku, "price"]
    cost = cost_df.loc[sku, "cost"]
    salvage = cost_df.loc[sku, "salvage"]
    greedy_profit += compute_daily_profit(y, q, price, cost, salvage).sum()

improvement_pct = (greedy_profit - prop_profit) / abs(prop_profit) * 100

budget_result = pd.DataFrame({
    "method": ["proportional", "greedy"],
    "portfolio_profit": [prop_profit, greedy_profit]
})

print("Unconstrained total procurement cost:", round(total_cost_unconstrained, 2))
print("Budget (75%):", round(budget, 2))
display(budget_result)
print("Improvement % (greedy vs proportional):", round(improvement_pct, 4))

*Your answer:*In this case, greedy actually does worse than proportional scaling. I think the reason is that the greedy rule is too simplified, since it ranks SKUs mainly by margin per cost and does not fully preserve the original order pattern. Proportional scaling may work better here because it keeps the same order mix as the unconstrained plan and just shrinks it to fit the budget.

### 4c. Dollar Value Decomposition (10 points)

Run the full pipeline **four** ways to decompose the profit improvement into prediction quality vs. loss function choice:

| Config | Mean Model | Order Quantity Method |
|--------|-----------|----------------------|
| (i)   | Global baseline (3a) | Normal approximation |
| (ii)  | Your alternative model (3b) | Normal approximation |
| (iii) | Global baseline (3a) | Quantile grid |
| (iv)  | Your alternative model (3b) | Quantile grid |

**Implementation notes:**
- For configs (i) and (ii), train a variance model (global GBDT on absolute residuals) using the corresponding mean model, then apply $Q^* = \hat{\mu} + z_{\tau^*} \cdot \hat{\sigma}$.
- For config (iii), reuse the quantile grid from Part 3c (which used `global_features`).
- For config (iv), re-train the quantile grid using the same model structure as your alternative in 3b. For example, if 3b used tier models, train separate quantile grids per tier; if 3b used cluster models, train separate quantile grids per cluster.

Report portfolio profit for each configuration in a 2x2 table:

|  | Normal Approx. | Quantile Grid |
|--|---------------|---------------|
| Global baseline | (i) | (iii) |
| Alternative model | (ii) | (iv) |

Compute:
- **(A)** Gain from better prediction = (ii) minus (i)
- **(B)** Gain from quantile regression = (iii) minus (i)
- **(C)** Combined gain = (iv) minus (i)

💻 **Code required**

📊 **Report:** 2x2 profit table, gains (A), (B), (C)

✍️ **Written answer** (3-5 sentences): Is the total improvement mostly from better prediction or from switching to the right loss function? Is the combined gain (C) larger or smaller than (A) + (B)? What does this tell you about the interaction between prediction quality and decision method?

In [ ]:
# YOUR CODE HERE
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from scipy.stats import norm

decomp_df = eval_df.copy()
train_alt = train.copy()

if "volume_tier" not in train_alt.columns:
    train_alt = train_alt.merge(
        sku_features[["volume_tier"]],
        left_on="item_id",
        right_index=True,
        how="left"
    )

if "volume_tier" not in decomp_df.columns:
    decomp_df = decomp_df.merge(
        sku_features[["volume_tier"]],
        left_on="item_id",
        right_index=True,
        how="left"
    )

train_alt = train_alt.dropna(subset=local_features + [TARGET, "volume_tier"]).copy()
decomp_df = decomp_df.dropna(subset=local_features + [TARGET, "volume_tier"]).copy()

# (ii) alternative mean model + normal approximation
train_alt["mu_alt"] = np.nan
for tier in ["high", "mid", "low"]:
    idx = train_alt["volume_tier"] == tier
    train_alt.loc[idx, "mu_alt"] = np.clip(
        tier_models[tier].predict(train_alt.loc[idx, local_features]),
        0,
        None
    )

train_alt["abs_resid_alt"] = np.abs(train_alt[TARGET] - train_alt["mu_alt"])

var_models_alt = {}
for tier in ["high", "mid", "low"]:
    tr_t = train_alt[train_alt["volume_tier"] == tier].copy()

    vm = lgb.LGBMRegressor(
        objective="regression",
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_estimators=100,
        verbose=-1
    )
    vm.fit(tr_t[local_features], tr_t["abs_resid_alt"])
    var_models_alt[tier] = vm

decomp_df["mu_alt"] = np.nan
decomp_df["sigma_alt"] = np.nan

for tier in ["high", "mid", "low"]:
    idx = decomp_df["volume_tier"] == tier
    decomp_df.loc[idx, "mu_alt"] = np.clip(
        tier_models[tier].predict(decomp_df.loc[idx, local_features]),
        0,
        None
    )
    decomp_df.loc[idx, "sigma_alt"] = np.clip(
        var_models_alt[tier].predict(decomp_df.loc[idx, local_features]),
        0,
        None
    )

decomp_df["tau_star"] = decomp_df["item_id"].map(cost_df["tau_star"])
decomp_df["z_tau"] = decomp_df["tau_star"].apply(norm.ppf)

decomp_df["q_alt_normal"] = np.round(
    np.clip(decomp_df["mu_alt"] + decomp_df["z_tau"] * decomp_df["sigma_alt"], 0, None)
).astype(int)

# (iv) alternative mean structure + quantile grid
taus = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

for tau in taus:
    decomp_df[f"alt_q_{tau:.1f}"] = np.nan

for tier in ["high", "mid", "low"]:
    tr_t = train_alt[train_alt["volume_tier"] == tier].copy()
    te_t = decomp_df[decomp_df["volume_tier"] == tier].copy()

    for tau in taus:
        model = lgb.LGBMRegressor(
            objective="quantile",
            alpha=tau,
            learning_rate=0.05,
            num_leaves=15,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_estimators=100,
            verbose=-1
        )
        model.fit(tr_t[local_features], tr_t[TARGET])
        decomp_df.loc[te_t.index, f"alt_q_{tau:.1f}"] = np.clip(
            model.predict(te_t[local_features]),
            0,
            None
        )

decomp_df["q_alt_grid"] = np.nan

for sku, part in decomp_df.groupby("item_id", sort=False):
    qmat = part[[f"alt_q_{tau:.1f}" for tau in taus]].values
    fixed = np.zeros_like(qmat)

    for i in range(qmat.shape[0]):
        iso = IsotonicRegression(increasing=True, out_of_bounds="clip")
        fixed[i, :] = iso.fit_transform(taus, qmat[i, :])

    tau_star = cost_df.loc[sku, "tau_star"]
    q_interp = []

    for i in range(fixed.shape[0]):
        q_interp.append(np.interp(tau_star, taus, fixed[i, :]))

    q_interp = np.round(np.clip(q_interp, 0, None)).astype(int)
    decomp_df.loc[part.index, "q_alt_grid"] = q_interp

def total_profit_from_orders(df, order_col):
    total = 0.0
    for sku, part in df.groupby("item_id"):
        y = part[TARGET].values
        q = part[order_col].values
        price = cost_df.loc[sku, "price"]
        cost = cost_df.loc[sku, "cost"]
        salvage = cost_df.loc[sku, "salvage"]
        total += compute_daily_profit(y, q, price, cost, salvage).sum()
    return total

profit_i = total_profit_from_orders(decomp_df, "q_normal")
profit_ii = total_profit_from_orders(decomp_df, "q_alt_normal")
profit_iii = total_profit_from_orders(decomp_df, "q_star_grid")
profit_iv = total_profit_from_orders(decomp_df, "q_alt_grid")

profit_2x2 = pd.DataFrame(
    {
        "Normal Approx.": [profit_i, profit_ii],
        "Quantile Grid": [profit_iii, profit_iv]
    },
    index=["Global baseline", "Alternative model"]
)

A = profit_ii - profit_i
B = profit_iii - profit_i
C = profit_iv - profit_i

display(profit_2x2)

gains = pd.DataFrame({
    "gain": ["A: better prediction", "B: quantile regression", "C: combined gain"],
    "value": [A, B, C]
})
display(gains)

Most of the improvement comes from switching to the quantile-based decision method rather than from improving the mean prediction alone, since B is much larger than A. The combined gain C is smaller than A + B, so the two effects are not additive. This suggests there is interaction between prediction quality and decision method: once the ordering rule is better aligned with the newsvendor objective, the extra value of changing the mean model becomes smaller.

---
## Part 5: Portfolio Risk-Return Analysis (10 points)

The newsvendor portfolio is analogous to a financial investment portfolio. Each SKU is an "asset" with an expected return (daily profit) and risk (profit variability). Unlike stocks, you cannot buy an arbitrary fraction of a SKU: you either stock it at $Q_i^*$ or not at all. This makes it a **binary portfolio selection** problem.

### 5a. Per-SKU Risk-Return Profile (2 points)

For each SKU under **unconstrained Method 2 orders** (from Part 3c), use the **test period** actuals to compute:
- $\mu_i$: mean daily profit (use `compute_daily_profit` to get per-day profits, then take the mean)
- $\sigma_i$: standard deviation of daily profit (standard deviation of the same per-day profits)
- $c_i$: total procurement cost over the test period (sum of $Q_i^* \times \text{cost}_i$ across all test days)

These quantities treat the 28-day test period as the evaluation window for each SKU's risk-return profile.

Produce a risk-return scatter plot: $\sigma_i$ (x-axis) vs. $\mu_i$ (y-axis), with bubble size proportional to $c_i$.

💻 **Code required**

📊 **Report:** risk-return scatter plot

In [ ]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

risk_rows = []

for sku, part in eval_df.groupby("item_id"):
    y = part[TARGET].values
    q = part["q_star_grid"].values

    price = cost_df.loc[sku, "price"]
    cost = cost_df.loc[sku, "cost"]
    salvage = cost_df.loc[sku, "salvage"]

    daily_profit = compute_daily_profit(y, q, price, cost, salvage)

    mu_i = daily_profit.mean()
    sigma_i = daily_profit.std(ddof=1)
    c_i = (q * cost).sum()

    risk_rows.append({
        "sku": sku,
        "mu_i": mu_i,
        "sigma_i": sigma_i,
        "c_i": c_i
    })

risk_df = pd.DataFrame(risk_rows)

plt.figure(figsize=(8, 6))
bubble_size = 20 + 180 * (risk_df["c_i"] / risk_df["c_i"].max())

plt.scatter(
    risk_df["sigma_i"],
    risk_df["mu_i"],
    s=bubble_size,
    alpha=0.6
)

plt.xlabel("sigma_i")
plt.ylabel("mu_i")
plt.title("Per-SKU risk-return profile")
plt.tight_layout()
plt.show()

risk_df.head()

### 5b. Random Portfolios and Efficient Frontier (5 points)

Use the per-SKU $\mu_i$, $\sigma_i$, and $c_i$ vectors computed in Part 5a. Set the budget $B$ at **75%** of the total unconstrained procurement cost: $B = 0.75 \times \sum_i c_i$.

**(i) Random portfolios.** Generate 500 random feasible portfolios. For each:
- Randomly shuffle the SKU list
- Add SKUs sequentially (at their full $Q_i^*$) until the remaining budget cannot accommodate the next SKU's $c_i$
- Compute portfolio mean daily profit: $\mu_p = \sum_i z_i \mu_i$
- Compute portfolio standard deviation: $\sigma_p = \sqrt{\sum_i z_i \sigma_i^2}$ (assuming independence across SKUs)
- where $z_i \in \{0, 1\}$ indicates whether SKU $i$ is included

Scatter these 500 points on the $(\sigma_p,\, \mu_p)$ plane.

**(ii) Efficient frontier.** Trace the efficient frontier by sweeping over a risk-aversion parameter $\lambda$. For each $\lambda$, solve the binary mean-variance problem:

$$\max_{z} \sum_i z_i \mu_i \;-\; \lambda \sum_i z_i \sigma_i^2 \quad \text{s.t.} \quad \sum_i z_i c_i \leq B, \quad z_i \in \{0, 1\}$$

Under the independence assumption, the objective is separable: each SKU contributes $z_i(\mu_i - \lambda \sigma_i^2)$ to the objective and $z_i c_i$ to the constraint. This is a **0-1 knapsack** with item values $v_i(\lambda) = \mu_i - \lambda \sigma_i^2$ and weights $c_i$. For each $\lambda$, solve the knapsack (e.g., greedy by value-to-weight ratio, or dynamic programming for exact solution), then compute $(\sigma_p, \mu_p)$ of the selected portfolio.

Sweep $\lambda$ over a range (e.g., 0 to a value where no SKU has positive adjusted value) and overlay the frontier on the random portfolio scatter.

💻 **Code required**

📊 **Report:** scatter of random portfolios with efficient frontier overlaid

*Hint:* With the independence assumption, the knapsack for each $\lambda$ can be solved greedily by ranking SKUs by $(\mu_i - \lambda \sigma_i^2) / c_i$ (including only SKUs with positive numerator) and adding until the budget is exhausted. This is the same logic as the greedy allocation in Part 4b, just with a different ranking criterion.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

prof = risk_df.copy()
prof = prof[prof["c_i"] > 0].copy()

B = 0.75 * prof["c_i"].sum()

mu = prof["mu_i"].values
sigma = prof["sigma_i"].values
cost = prof["c_i"].values

rng = np.random.default_rng(42)
rand_points = []

for _ in range(500):
    order = rng.permutation(len(prof))
    used = 0.0
    z = np.zeros(len(prof), dtype=int)

    for j in order:
        if used + cost[j] <= B:
            z[j] = 1
            used += cost[j]

    mu_p = np.sum(z * mu)
    sigma_p = np.sqrt(np.sum(z * (sigma ** 2)))

    rand_points.append({
        "mu_p": mu_p,
        "sigma_p": sigma_p
    })

rand_df = pd.DataFrame(rand_points)

lam_grid = np.linspace(0, 10, 400)
frontier = []

for lam in lam_grid:
    value = mu - lam * (sigma ** 2)
    ratio = np.where((value > 0) & (cost > 0), value / cost, -np.inf)

    order = np.argsort(-ratio)
    used = 0.0
    z = np.zeros(len(prof), dtype=int)

    for j in order:
        if ratio[j] == -np.inf:
            continue
        if used + cost[j] <= B:
            z[j] = 1
            used += cost[j]

    mu_p = np.sum(z * mu)
    sigma_p = np.sqrt(np.sum(z * (sigma ** 2)))

    frontier.append({
        "lambda": lam,
        "mu_p": mu_p,
        "sigma_p": sigma_p
    })

frontier_df = pd.DataFrame(frontier)
frontier_df = frontier_df.drop_duplicates(subset=["mu_p", "sigma_p"]).sort_values("sigma_p")

plt.figure(figsize=(8, 6))
plt.scatter(rand_df["sigma_p"], rand_df["mu_p"], alpha=0.4, label="Random portfolios")
plt.plot(frontier_df["sigma_p"], frontier_df["mu_p"], linewidth=2, label="Efficient frontier")
plt.xlabel("sigma_p")
plt.ylabel("mu_p")
plt.title("Random portfolios and efficient frontier")
plt.legend()
plt.tight_layout()
plt.show()

### 5c. Tangency Portfolio (3 points)

Identify the point on the efficient frontier with the highest $\mu_p / \sigma_p$ ratio (the "tangency portfolio" or maximum Sharpe ratio point). Report which SKUs are included and which are excluded.

💻 **Code required**

📊 **Report:** tangency portfolio Sharpe ratio, list of included/excluded SKUs

✍️ **Written answer** (3-5 sentences): What characteristics (volume, margin, demand variability) distinguish the included SKUs from the excluded ones? In standard financial portfolio theory, you can hold any fraction of an asset, producing a smooth parabolic frontier. Here each SKU is all-or-nothing. How does this discreteness affect the shape of the frontier compared to the Markowitz parabola?

In [ ]:
# YOUR CODE HERE
tangency_df = frontier_df.copy()
tangency_df = tangency_df[tangency_df["sigma_p"] > 0].copy()
tangency_df["sharpe_like"] = tangency_df["mu_p"] / tangency_df["sigma_p"]

best_idx = tangency_df["sharpe_like"].idxmax()
best_row = tangency_df.loc[best_idx]

best_lambda = best_row["lambda"]

prof = risk_df.copy()
prof = prof[prof["c_i"] > 0].copy()

mu = prof["mu_i"].values
sigma = prof["sigma_i"].values
cost = prof["c_i"].values
sku_list = prof["sku"].values

B = 0.75 * prof["c_i"].sum()

value = mu - best_lambda * (sigma ** 2)
ratio = np.where((value > 0) & (cost > 0), value / cost, -np.inf)

order = np.argsort(-ratio)
used = 0.0
z = np.zeros(len(prof), dtype=int)

for j in order:
    if ratio[j] == -np.inf:
        continue
    if used + cost[j] <= B:
        z[j] = 1
        used += cost[j]

included_skus = sku_list[z == 1].tolist()
excluded_skus = sku_list[z == 0].tolist()

print("Tangency portfolio")
print("lambda =", round(best_lambda, 6))
print("mu_p =", round(best_row["mu_p"], 6))
print("sigma_p =", round(best_row["sigma_p"], 6))
print("mu_p / sigma_p =", round(best_row["sharpe_like"], 6))
print("included SKUs =", len(included_skus))
print("excluded SKUs =", len(excluded_skus))

tangency_summary = pd.DataFrame({
    "metric": ["lambda", "mu_p", "sigma_p", "mu_p / sigma_p", "included_count", "excluded_count"],
    "value": [
        best_lambda,
        best_row["mu_p"],
        best_row["sigma_p"],
        best_row["sharpe_like"],
        len(included_skus),
        len(excluded_skus)
    ]
})

display(tangency_summary)

print("\nIncluded SKUs:")
print(included_skus)

print("\nExcluded SKUs:")
print(excluded_skus)

*Your answer:*The SKUs included in the tangency portfolio are likely the ones with better return relative to both risk and budget cost, so they offer stronger profit-to-risk tradeoffs than the excluded SKUs. The excluded SKUs are more likely to have lower mean profit, higher variability, or weaker value relative to their procurement cost. Unlike the smooth Markowitz parabola in standard finance, the frontier here is more jagged because this is a discrete all-or-nothing selection problem. Since each SKU can only be fully included or excluded, portfolio changes happen in jumps rather than through continuous weight adjustments.

---
## Part 6: Reflection (10 points)

Answer each question in 3-5 sentences.

### 6a. Prediction Quality vs. Decision Quality (4 points)

Look at the 2x2 decomposition in Part 4c. Did RMSE differences between the global baseline and your alternative model translate into proportional profit differences? Is it possible for a model with higher RMSE to produce higher newsvendor profit? Explain the mechanism, using your results as evidence.

✍️ **Written answer**

*Your answer:*The RMSE differences did not translate proportionally into profit differences. In Part 4c, the gain from better prediction alone was only about 349, while the gain from switching to the quantile-based decision method was much larger at about 2214. This shows that decision quality depends not only on forecast accuracy in the RMSE sense, but also on whether the model is aligned with the newsvendor objective. So yes, a model with higher RMSE can still produce higher newsvendor profit if it gives order quantities that are better calibrated to the relevant quantiles and cost asymmetry.

### 6b. Distributional Assumptions (3 points)

Based on your coverage scatter plot in Part 4a, for what fraction of SKUs is the Normal approximation adequate (realized coverage within $\pm 0.05$ of target $\tau^*$)? What observable SKU characteristic best predicts where normality fails? If you could only use one method (Normal or quantile grid) for the entire portfolio, which would you choose and why?

✍️ **Written answer**

*Your answer:*The Normal approximation is adequate for about 11.2% of SKUs if adequacy is defined as realized coverage within ±0.05 of the target tau*. It fails more often for sparse SKUs, especially those with higher zero fraction and lower volume, which is consistent with the larger average coverage error in the low-volume tier and the positive correlation between zero fraction and calibration error. If I had to use only one method for the whole portfolio, I would choose the quantile grid. It was adequate for a larger fraction of SKUs, produced higher total portfolio profit, and is more reliable when demand is intermittent and far from Normal.

### 6c. Deployment Feedback Loop (3 points)

Suppose you deploy your quantile grid order quantities in production. Each day, the store orders $Q_i^*$ units for each SKU. On days when true demand exceeds the order quantity, the store stocks out and you observe sales $= Q_i^*$, not the true demand. Next month, you retrain the quantile GBDT on the new sales data that includes these stockout days.

How is the new training set biased compared to the historical data used in this assignment? Which SKUs are most affected? Propose one practical approach to mitigate this problem without modifying the quantile GBDT model itself.

✍️ **Written answer**

*Your answer:*The new training data would be downward biased because on stockout days the recorded sales are censored at the order quantity instead of reflecting the true underlying demand. This means the retrained model would learn from artificially low outcomes, so it would tend to underpredict demand and reinforce under-ordering over time. The SKUs most affected are the ones with frequent stockouts, especially high-demand or highly variable items where true demand often exceeds the order quantity. A practical mitigation is to flag stockout days in the data pipeline and either exclude them from retraining or add a censoring indicator / imputed demand estimate before training, without changing the quantile GBDT itself.

---
## Submission Checklist

Before submitting, verify:

- [ ] All code cells run **top-to-bottom** without errors (Kernel → Restart & Run All)
- [ ] All tables, plots, and figures are visible in the notebook output
- [ ] All written-answer cells are filled in
- [ ] You have not modified the data loading, cost structure, or helper function cells
- [ ] Parts 1-2 use the designated focal SKUs (`sku_high`, `sku_sparse`)
- [ ] Part 4a includes a variance model for the Normal approximation (Method 1)
- [ ] Part 4c reports the full 2x2 decomposition with gains (A), (B), (C), with config (iv) using re-trained quantile models matching your 3b model structure
- [ ] Part 5 includes both random portfolio scatter and efficient frontier overlay

---
## Grading Rubric

| Part | Points | Key Criteria |
|------|--------|-------------|
| **1. Single-SKU Forecasting** | **15** | |
| 1a. AR baseline | 5 | Correct RMSE, fitted-vs-actual plot, residual interpretation |
| 1b. GBDT comparison | 5 | Correct RMSE, ACF plots, reasoned comparison (not assuming GBDT wins) |
| 1c. Flexibility analysis | 5 | Correct comparison table, nuanced explanation that accounts for the possibility that simpler models suffice |
| **2. Single-SKU Quantile** | **15** | |
| 2a. Normal approximation | 5 | Correct $Q^*$ computation, coverage, profit |
| 2b. Quantile GBDT | 5 | Correct fitting, overlay plot, identification of disagreement patterns |
| 2c. Residual diagnostics | 5 | Histogram + Normal overlay, Shapiro-Wilk test, connection between normality failure and coverage |
| **3. Multi-SKU Portfolio** | **30** | |
| 3a. Global model | 5 | Correct RMSE by tier |
| 3b. Forecast improvement | 10 | Working alternative model, scatter diagnostic, honest assessment of whether it helps |
| 3c. Quantile grid | 15 | Correct grid training, isotonic crossing fix, interpolation, quantile fan figure, written understanding of zero-flat quantiles |
| **4. Evaluation & Decomposition** | **20** | |
| 4a. Coverage and profit | 5 | Correct coverage scatter, profit by tier |
| 4b. Budget allocation | 5 | Working proportional and greedy implementations, profit comparison |
| 4c. Dollar value decomposition | 10 | Complete 2x2 table, correct gains (A)(B)(C), insightful interpretation of interaction |
| **5. Risk-Return Analysis** | **10** | |
| 5a. Per-SKU profile | 2 | Correct risk-return scatter with bubble sizes |
| 5b. Random portfolios + frontier | 5 | 500 random portfolios via sequential filling, correct knapsack sweep, clean overlay figure |
| 5c. Tangency portfolio | 3 | Correct identification, included/excluded analysis, discussion of discreteness vs. Markowitz |
| **6. Reflection** | **10** | |
| 6a. Prediction vs. decision quality | 4 | Nuanced reasoning about RMSE-profit relationship |
| 6b. Distributional assumptions | 3 | Quantified Normal adequacy, identified predictive characteristic |
| 6c. Deployment feedback loop | 3 | Correct identification of downward bias, identification of most-affected SKUs, practical mitigation at the data level |
| **Total** | **100** | |